# 09 — Tracing & Events

This notebook demonstrates the orqest observability primitives: tracing agent execution with `JSONTracer` and reacting to runtime events with `EventBus`.

**Use case:** Observing a multi-agent pipeline in real-time — seeing exactly what happened, when, and how long each step took.

**Prerequisites:**
- A `.env` file in the project root (or environment variables) with:
  ```
  LLM_API_KEY=your_api_key
  LLM_MODEL=openai:gpt-4.1
  ```

## 1. Setup

Load the orqest config and create a `JSONTracer` (in-memory tracer with JSON export) and an `EventBus` (in-process pub/sub for agent events).

In [ ]:
from orqest import load_config
from orqest.observability import JSONTracer, AgentEvent, EventBus

config = load_config()
print(f"Model:    {config.llm_model}")
print(f"API key:  {config.llm_api_key[:8]}...")

tracer = JSONTracer()
bus = EventBus()
print(f"Tracer:   {type(tracer).__name__}")
print(f"EventBus: {type(bus).__name__}")

## 2. Basic tracing

Create spans manually to understand the tracing primitives. Each span gets a unique `span_id` and belongs to a `trace_id`. Child spans inherit their parent's `trace_id`.

In [ ]:
import asyncio

# Root span — starts a new trace
root = tracer.start_span("my_pipeline", agent_name="orchestrator")
print(f"Root span:   trace_id={root.trace_id[:12]}...  span_id={root.span_id[:12]}...  parent=None")

# Child span — inherits the trace_id from root
child1 = tracer.start_span("step_1_extract", agent_name="extractor", parent=root)
print(f"Child span:  trace_id={child1.trace_id[:12]}...  span_id={child1.span_id[:12]}...  parent={child1.parent_span_id[:12]}...")

# Simulate some work
await asyncio.sleep(0.05)
tracer.end_span(child1, status="ok", attributes={"tokens": 150})

# Another child span
child2 = tracer.start_span("step_2_transform", agent_name="transformer", parent=root)
await asyncio.sleep(0.03)
tracer.end_span(child2, status="ok", attributes={"tokens": 200})

# End the root span
tracer.end_span(root, status="ok")

print(f"\nAll spans share the same trace_id: {root.trace_id[:12]}...")
print(f"child1 duration: {child1.duration_ms:.1f} ms")
print(f"child2 duration: {child2.duration_ms:.1f} ms")
print(f"root   duration: {root.duration_ms:.1f} ms")

## 3. Traced agent

Wrap a real agent run with tracer spans. We manually call `start_span` before the agent runs and `end_span` after it completes.

In [ ]:
from pydantic import BaseModel, Field
from orqest.agents import BaseAgent, GlobalState


class BriefOutput(BaseModel):
    """Short answer from the agent."""
    answer: str = Field(description="A one-sentence answer")


class BriefAgent(BaseAgent[GlobalState, BriefOutput]):
    def __init__(self, model: str, api_key: str):
        super().__init__(
            agent_name="brief_agent",
            system_prompt="Answer in exactly one sentence.",
            output_type=BriefOutput,
            model=model,
            api_key=api_key,
        )

    async def _run_implementation(self, state: GlobalState, **kwargs) -> BriefOutput:
        user_message = state.get_latest_message("user")
        result = await self.call_model(user_message, state)
        return result.output


# Clear previous spans for a clean demo
tracer.clear()

agent = BriefAgent(model=config.llm_model, api_key=config.llm_api_key)
state = GlobalState()
state.add_message("user", "What is the capital of France?")

# Trace the agent run
span = tracer.start_span("brief_agent.run", agent_name="brief_agent")
try:
    output = await agent.run(state)
    tracer.end_span(span, status="ok", attributes={"answer_length": len(output.answer)})
except Exception as exc:
    tracer.end_span(span, status="error", attributes={"error": str(exc)})
    raise

print(f"Answer: {output.answer}")
print(f"Span:   name={span.name}, duration={span.duration_ms:.1f} ms, status={span.status}")

## 4. Export traces

`JSONTracer.export_json()` serializes all spans to a list of JSON-safe dicts. This is the data you would send to your observability backend.

In [ ]:
import json

exported = tracer.export_json()
print(f"Exported {len(exported)} span(s):\n")
print(json.dumps(exported, indent=2, default=str))

## 5. Event bus basics

The `EventBus` is a pub/sub dispatcher. Subscribe handlers to specific event types, then emit events. Handlers are called in order.

In [ ]:
event_log: list[str] = []


def on_agent_start(event: AgentEvent) -> None:
    """Sync handler for agent_start events."""
    msg = f"[{event.event_type}] agent={event.agent_name} data={event.data}"
    event_log.append(msg)
    print(f"  Handler fired: {msg}")


def on_agent_complete(event: AgentEvent) -> None:
    """Sync handler for agent_complete events."""
    msg = f"[{event.event_type}] agent={event.agent_name} data={event.data}"
    event_log.append(msg)
    print(f"  Handler fired: {msg}")


bus.subscribe("agent_start", on_agent_start)
bus.subscribe("agent_complete", on_agent_complete)

# Emit events
print("Emitting agent_start:")
await bus.emit(AgentEvent(event_type="agent_start", agent_name="extractor", data={"step": 1}))

print("\nEmitting agent_complete:")
await bus.emit(AgentEvent(event_type="agent_complete", agent_name="extractor", data={"tokens": 150}))

print(f"\nEvent log has {len(event_log)} entries")

## 6. Sync and async handlers

The `EventBus` supports both synchronous and asynchronous handlers. It detects coroutines automatically and awaits them.

In [ ]:
async_log: list[str] = []


def sync_handler(event: AgentEvent) -> None:
    """A plain synchronous handler."""
    async_log.append(f"sync: {event.event_type}")
    print(f"  sync_handler fired for {event.event_type}")


async def async_handler(event: AgentEvent) -> None:
    """An async handler — the bus awaits it automatically."""
    await asyncio.sleep(0.01)  # simulate async work
    async_log.append(f"async: {event.event_type}")
    print(f"  async_handler fired for {event.event_type}")


# Fresh bus for this demo
demo_bus = EventBus()
demo_bus.subscribe("test_event", sync_handler)
demo_bus.subscribe("test_event", async_handler)

print("Emitting test_event:")
await demo_bus.emit(AgentEvent(event_type="test_event", agent_name="demo"))

print(f"\nLog: {async_log}")

## 7. Error resilience

A broken handler cannot crash the event bus. Exceptions are logged at WARNING level and swallowed — other handlers still run. This is the same fire-and-forget pattern used by `HookRunner`.

In [ ]:
resilience_log: list[str] = []


def broken_handler(event: AgentEvent) -> None:
    """This handler always raises an exception."""
    raise ValueError("I'm broken!")


def healthy_handler(event: AgentEvent) -> None:
    """This handler works fine."""
    resilience_log.append(f"healthy: {event.event_type}")
    print(f"  healthy_handler fired for {event.event_type}")


error_bus = EventBus()
error_bus.subscribe("crash_test", broken_handler)
error_bus.subscribe("crash_test", healthy_handler)

print("Emitting crash_test (broken_handler will fail, healthy_handler should still run):")
await error_bus.emit(AgentEvent(event_type="crash_test", agent_name="test"))

print(f"\nHealthy handler ran: {len(resilience_log) == 1}")
print(f"No crash: True")

## 8. Pipeline observability

Now let's combine tracing and events with a real 2-step pipeline. We create a pipeline-level span, child spans for each step, and emit events at each lifecycle point. This shows how you would instrument a production pipeline.

In [ ]:
from orqest.orchestration import Pipeline, FunctionStep


class ExtractOutput(BaseModel):
    """Extracted key facts."""
    facts: list[str] = Field(description="Key facts extracted from the text")


class SynthesisOutput(BaseModel):
    """Synthesized summary."""
    summary: str = Field(description="A synthesis of the extracted facts")


class ExtractAgent(BaseAgent[GlobalState, ExtractOutput]):
    def __init__(self, model: str, api_key: str):
        super().__init__(
            agent_name="extractor",
            system_prompt="Extract 3 key facts from the given text. Be concise.",
            output_type=ExtractOutput,
            model=model,
            api_key=api_key,
        )

    async def _run_implementation(self, state: GlobalState, **kwargs) -> ExtractOutput:
        user_message = state.get_latest_message("user")
        result = await self.call_model(user_message, state)
        return result.output


class SynthesisAgent(BaseAgent[GlobalState, SynthesisOutput]):
    def __init__(self, model: str, api_key: str):
        super().__init__(
            agent_name="synthesizer",
            system_prompt="Synthesize the given facts into a single cohesive paragraph.",
            output_type=SynthesisOutput,
            model=model,
            api_key=api_key,
        )

    async def _run_implementation(self, state: GlobalState, **kwargs) -> SynthesisOutput:
        user_message = state.get_latest_message("user")
        result = await self.call_model(user_message, state)
        return result.output


# Create agents
extractor = ExtractAgent(model=config.llm_model, api_key=config.llm_api_key)
synthesizer = SynthesisAgent(model=config.llm_model, api_key=config.llm_api_key)

print("Agents created: extractor, synthesizer")

In [ ]:
# Fresh tracer and bus for the pipeline demo
pipeline_tracer = JSONTracer()
pipeline_bus = EventBus()
pipeline_events: list[str] = []


def log_pipeline_event(event: AgentEvent) -> None:
    """Log all pipeline events."""
    pipeline_events.append(f"{event.event_type}: {event.agent_name}")
    print(f"  EVENT: [{event.event_type}] agent={event.agent_name} span_id={event.span_id[:12] if event.span_id else 'none'}...")


pipeline_bus.subscribe_all(log_pipeline_event)

# Input text
input_text = (
    "Rust's ownership system eliminates data races at compile time. The borrow checker "
    "enforces strict rules about mutable and immutable references. This means Rust "
    "programs are memory-safe without a garbage collector, making them suitable for "
    "systems programming, embedded devices, and WebAssembly."
)

# --- Run the pipeline with tracing and events ---

# 1. Pipeline-level span
pipeline_span = pipeline_tracer.start_span("extract_and_synthesize", agent_name="pipeline")
await pipeline_bus.emit(AgentEvent(
    event_type="pipeline_start", agent_name="pipeline",
    span_id=pipeline_span.span_id, trace_id=pipeline_span.trace_id,
))

# 2. Step 1: Extract
step1_span = pipeline_tracer.start_span("step_extract", agent_name="extractor", parent=pipeline_span)
await pipeline_bus.emit(AgentEvent(
    event_type="step_start", agent_name="extractor",
    span_id=step1_span.span_id, trace_id=step1_span.trace_id, data={"step": 1},
))

extract_state = GlobalState()
extract_state.add_message("user", input_text)
extract_output = await extractor.run(extract_state)

pipeline_tracer.end_span(step1_span, status="ok", attributes={"fact_count": len(extract_output.facts)})
await pipeline_bus.emit(AgentEvent(
    event_type="step_complete", agent_name="extractor",
    span_id=step1_span.span_id, trace_id=step1_span.trace_id,
    data={"facts": len(extract_output.facts)},
))

print(f"\nExtracted facts:")
for f in extract_output.facts:
    print(f"  - {f}")

# 3. Step 2: Synthesize (using extracted facts as input)
step2_span = pipeline_tracer.start_span("step_synthesize", agent_name="synthesizer", parent=pipeline_span)
await pipeline_bus.emit(AgentEvent(
    event_type="step_start", agent_name="synthesizer",
    span_id=step2_span.span_id, trace_id=step2_span.trace_id, data={"step": 2},
))

synth_state = GlobalState()
facts_text = "\n".join(f"- {f}" for f in extract_output.facts)
synth_state.add_message("user", f"Synthesize these facts:\n{facts_text}")
synth_output = await synthesizer.run(synth_state)

pipeline_tracer.end_span(step2_span, status="ok", attributes={"summary_length": len(synth_output.summary)})
await pipeline_bus.emit(AgentEvent(
    event_type="step_complete", agent_name="synthesizer",
    span_id=step2_span.span_id, trace_id=step2_span.trace_id,
))

# 4. End pipeline
pipeline_tracer.end_span(pipeline_span, status="ok")
await pipeline_bus.emit(AgentEvent(
    event_type="pipeline_complete", agent_name="pipeline",
    span_id=pipeline_span.span_id, trace_id=pipeline_span.trace_id,
))

print(f"\nSynthesis: {synth_output.summary}")

## 9. Assessment

Visualize the trace hierarchy and summarize the pipeline execution.

In [ ]:
# Visualize the trace hierarchy
spans = pipeline_tracer.get_spans()

print("Trace hierarchy:")
print(f"  trace_id: {spans[0].trace_id[:16]}...\n")
for s in spans:
    indent = "    " if s.parent_span_id else "  "
    arrow = "+-" if s.parent_span_id else ""
    duration = f"{s.duration_ms:.0f}ms" if s.duration_ms else "running"
    print(f"{indent}{arrow}[{s.name}] agent={s.agent_name} status={s.status} duration={duration}")
    if s.attributes:
        print(f"{indent}  attributes: {s.attributes}")

# Summary stats
print(f"\n--- Summary ---")
print(f"Total spans:  {len(spans)}")
print(f"Total events: {len(pipeline_events)}")
print(f"Pipeline duration: {pipeline_span.duration_ms:.0f} ms")
print(f"  Step 1 (extract):    {step1_span.duration_ms:.0f} ms")
print(f"  Step 2 (synthesize): {step2_span.duration_ms:.0f} ms")

## What's happening under the hood

1. **Zero-dependency design**: Both `JSONTracer` and `EventBus` are pure Python with no external dependencies. No OpenTelemetry, no Jaeger, no Prometheus — just dataclasses and lists. This keeps the framework lightweight and easy to embed.

2. **JSONTracer**: Stores `Span` objects in an in-memory list. Each span has a `trace_id` (shared by all spans in a trace) and a `span_id` (unique). Child spans inherit their parent's `trace_id` via the `parent` parameter. `export_json()` serializes everything to JSON-safe dicts you can send to any backend.

3. **Span lifecycle**: `start_span()` creates the span and records `started_at`. `end_span()` sets `ended_at`, computes `duration_ms`, and merges any extra attributes. The status field is either `"ok"` or `"error"`.

4. **EventBus fire-and-forget**: Handler errors are caught by `_safe_call()` and logged via loguru at WARNING level. They are never re-raised. This means a broken analytics handler cannot crash your agent pipeline — the same pattern used by `HookRunner` for tool lifecycle hooks.

5. **Sync + async handlers**: The bus calls `handler(event)` and checks if the result is a coroutine. If so, it awaits it. This lets you mix sync logging handlers with async database writers without any special registration.

6. **Correlation**: `AgentEvent` has optional `span_id` and `trace_id` fields. By passing the current span's IDs when emitting events, you can correlate events with their trace spans after the fact — joining the event stream with the trace tree.

7. **JSON export format**: The exported spans are plain dicts with ISO 8601 timestamps. You can pipe them into any observability backend (Datadog, Honeycomb, your own dashboard) with a simple HTTP POST.